# Paper Evaluation: Exact Dynamic Programming vs. Greedy Allocation

This notebook runs the reproducible empirical study for minimum-sensor zero-error field discrimination. It generates any missing study data, evaluates both methods, computes uncertainty intervals, and displays publication-ready figures.

## Research questions

1. How close is the **Greedy Gain-Based Approximation** to the optimum returned by **Exact Dynamic Programming**?
2. Is solution quality consistent across field models, numbers of fields `K`, and numbers of sensing locations `M`?
3. Do the conclusions reproduce under an independent random seed?
4. How sensitive are the conclusions to the physical phase-scale constant `c`?
5. How do the runtimes scale over the range where the exact optimum can still be computed?

The old 20-case `benchmark` remains useful only as a smoke test. It is not used as paper evidence.

## Pre-specified study design

| Dataset | Purpose | Design | Cases |
|---|---|---:|---:|
| `study-primary` | Main quality and runtime analysis | 4 models × 8 values of `K` × 3 values of `M` × 30 realizations | 2,880 |
| `study-replication` | Independent-seed confirmation | Same factorial design × 10 new realizations | 960 |
| `study-sensitivity` | Paired phase-scale analysis | 4 models × 3 values of `K` × 2 values of `M` × 3 values of `c` × 15 realizations | 1,080 |

The primary study uses `K = 3, …, 10`, `M ∈ {3, 5, 8}`, and `c = 1`. The four transparent synthetic field families are Gaussian, sinusoidal, two-harmonic Fourier, and cubic polynomial. Every seed and every generated field parameter is saved in the dataset manifest.

Thirty realizations per factorial cell give 360 primary cases at each `K` and 720 per field family. A convergence diagnostic checks whether the pooled estimates stabilize as realizations per cell increase from 5 to 30.

## Outcomes and uncertainty

The primary outcome is the approximation ratio `ρ = N_greedy / N_optimal`. The analysis also reports exact-optimum recovery, percentage sensor excess, the 95th percentile and maximum ratio, and runtime.

The theorem permits `ρ ≤ 1 + ln(K(K−1)/2)`. To show the empirical gap without a visually misleading scale, the figure reports the fraction of the theorem's *permitted excess* that was observed:

`(ρ − 1) / ln(K(K−1)/2)`.

Mean-ratio intervals use 2,000 reproducible bootstrap resamples within factorial cells, preserving the balanced design. Exact-match intervals use the 95% Wilson interval. The phase-scale experiment uses paired differences because the same generated field realization is evaluated at each `c`, with bootstrap resampling within model × `M` strata. A p-value against the approximation theorem is neither needed nor appropriate: the theorem is a deterministic worst-case upper bound, not a null hypothesis.

## Load the study pipeline

Reusable algorithms and analysis remain in `.py` files; this notebook is the readable experiment runner.

In [ ]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
candidates = [
    working_directory,
    working_directory / "min-sensor-zero-error",
    *(parent / "min-sensor-zero-error" for parent in working_directory.parents),
]
project_directory = next(
    candidate for candidate in candidates
    if (candidate / "evaluate_paper_studies.py").is_file()
)
if str(project_directory) not in sys.path:
    sys.path.insert(0, str(project_directory))

from evaluate_paper_studies import run_paper_evaluation
from IPython.display import Markdown, display

print(f"Study code: {project_directory}")

## Configure the run

`REPEATS = 1` is sufficient for solution-quality results because both solvers are deterministic. Before reporting runtime in the paper, use at least `REPEATS = 5` and run on an otherwise idle machine; this makes the full notebook slower but averages timing noise.

In [ ]:
REPEATS = 1
BOOTSTRAP_ITERATIONS = 2_000

## Run the complete evaluation

The first run creates any missing datasets. Every run evaluates all 4,920 cases with both Exact Dynamic Programming and the Greedy Gain-Based Approximation, then refreshes the statistical tables and figures. Expect roughly a minute on a typical laptop when `REPEATS = 1`; exact timing depends on the machine.

In [ ]:
analysis = run_paper_evaluation(
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)

## Main numerical findings

This text is generated directly from the current results, including the independent replication and sample-stability diagnostic.

In [ ]:
findings_path = project_directory / "output" / "paper-analysis" / "paper_findings.md"
display(Markdown(findings_path.read_text(encoding="utf-8")))

## Publication figures: solution quality

The first figure reports distributions rather than individual case labels: box limits are quartiles, whiskers are the 5th and 95th percentiles, exact-match uncertainty uses Wilson intervals, and the final panel directly measures how much of the worst-case permitted excess was observed. The ECDF reveals the complete quality distribution and differences among field families.

In [ ]:
display(analysis["figures"]["quality"])
display(analysis["figures"]["ecdf"])

## Robustness and computational diagnostics

Runtime curves show medians with interquartile bands on a logarithmic scale. Phase-scale effects are paired against `c = 1` and include bootstrap intervals. The convergence plot tests whether adding realizations materially changes the primary estimates.

In [ ]:
display(analysis["figures"]["runtime"])
display(analysis["figures"]["sensitivity"])
display(analysis["figures"]["convergence"])

## Saved paper artifacts

Vector PDF figures are intended for the manuscript; 300-dpi PNG versions are convenient for preview. CSV files preserve case-level metrics, grouped summaries, and the sample-convergence diagnostic.

In [ ]:
analysis_directory = project_directory / "output" / "paper-analysis"
for path in sorted(analysis_directory.iterdir()):
    print(path.name)

## Interpretation for the paper

Describe the conclusion as an empirical finding on the stated synthetic spatial-field distribution: the greedy method was much closer to the exact optimum than the theorem's worst-case allowance. Do **not** call it a stronger approximation guarantee or a universal average-case result.

Report the sampling design and parameter distributions so readers can judge external validity. The independent-seed replication, family-stratified ECDF, phase-scale sensitivity study, and sample-convergence diagnostic should be presented as robustness evidence. For a final paper, also state the processor, operating system, Python version, and timing repetitions used for the runtime figure.